In [1]:
import numpy as np
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

from splank import SoftPlankton, Flow, FluidPlankton

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Assembly creation

### Data 

In [2]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:        # DOF prefixes/names (extends default "dof")
    - x           # Recognized as a DOF prefix/name (e.g., "x0" in expressions)
param_names:      # Parameter prefixes/names (extends default "param" and "k")
    - myradius    # Recognized as a parameter prefix/name (e.g., `myradius` in expressions)

# Default Values (Optional)
defaults:
  myradius0: 1.   # Parameter: Listed in `param_names`
  myradius1: 1.
  k: 1            # Parameter: Auto-detected (default prefix "k")

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: myradius0                       
    mass: 1
    position: [-1, 0, 0]             
    force: [k*x, 0, 0]              

  - # Sphere 1 #################
    radius: myradius1                 
    mass: -1
    position: [1+x, 0, 0]       
    force: [-k*x, 0, 0]
"""

In [3]:
myplankton= SoftPlankton(yaml_data, verbose=False)

print(repr(myplankton))

SPHERE ASSEMBLY
  2 spheres
  1 degrees of freedom
  2 fixed parameters

Default values
  degrees of freedom dof: ['x'] = [ 0.]
  fixed parameters param: ['myradius0', 'myradius1'] = [ 1.  1.]

SPHERE 0
  radius: 1.0
  position: [-1.  0.  0.]
  orientation: [ 0.  0.  0.]
  force: [ 0.  0.  0.]
  torque: [ 0.  0.  0.]

SPHERE 1
  radius: 1.0
  position: [ 1.  0.  0.]
  orientation: [ 0.  0.  0.]
  force: [ 0.  0.  0.]
  torque: [ 0.  0.  0.]



### Useful plotting utility functions

In [4]:
def rotate_points(points, axis_angle):
    """
    Rotate an array of points using Rodrigues' rotation formula.
    """
    points = np.array(points)
    theta = np.linalg.norm(np.array(axis_angle))
    if theta < 1e-8:  # No rotation
        return points
    
    k = axis_angle / theta  # Unit rotation axis
    kx, ky, kz = k
    
    K = np.array([
        [0, -kz, ky],
        [kz, 0, -kx],
        [-ky, kx, 0]
    ])
    
    R = np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * np.dot(K, K)  # Rodrigues formula
    
    return np.dot(points, R.T)

In [5]:
def create_sphere(radius=1, center=(0, 0, 0), orientation=(0, 0, 0), resolution=32, checkered_scale=8):
    """
    Create a 3D checkered sphere with sharp transitions.
    """
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)

    x = radius * np.outer(np.cos(u), np.sin(v))
    y = radius * np.outer(np.sin(u), np.sin(v))
    z = radius * np.outer(np.ones(np.size(u)), np.cos(v))

    # Stack points and apply rotation
    points = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    rotated_points = rotate_points(points, orientation)

    x_rot = rotated_points[:, 0].reshape(x.shape) + center[0]
    y_rot = rotated_points[:, 1].reshape(y.shape) + center[1]
    z_rot = rotated_points[:, 2].reshape(z.shape) + center[2]

    # Generate sharp checkered pattern by controlling color block size
    checkered_u = (np.floor(np.linspace(0, resolution, resolution) / checkered_scale) % 2).astype(int)
    checkered_v = (np.floor(np.linspace(0, resolution, resolution) / checkered_scale) % 2).astype(int)
    colors = np.add.outer(checkered_u, checkered_v) % 2  # 0s and 1s for high contrast

    return x_rot, y_rot, z_rot, colors

In [6]:
def add_arrow(fig, start, direction, color, name):
    """
    Adds a 3D arrow (line + cone) to the figure.
    """
    end = start + direction
    cone_start = start + 0.75 * direction
    arrow_length = np.linalg.norm(direction)

    # Line (shaft)
    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]], y=[start[1], end[1]], z=[start[2], end[2]], 
        mode="lines", line=dict(color=color, width=5), name=f"{name}-axis"
    ))

    # Cone (arrowhead)
    fig.add_trace(go.Cone(
        x=[cone_start[0]], y=[cone_start[1]], z=[cone_start[2]],
        u=[direction[0]], v=[direction[1]], w=[direction[2]], 
        colorscale=[[0, color], [1, color]], showscale=False,
        sizemode="absolute", sizeref=0.333 * arrow_length  # Adjust arrowhead size
    ))

In [7]:
def update_figure(fig, assembly, dofs=None, params=None):
    """
    Efficiently update the existing 3D visualization of SphereAssembly.
    """
    dofs_array = np.array(dofs) if dofs is not None else assembly.dof_defaults
    params_array = np.array(params) if params is not None else assembly.param_defaults

    # Clear previous data while keeping the figure object
    fig.data = []

    # Add 3D coordinate arrows
    add_arrow(fig, start=np.array([0, 0, 0]), direction=np.array([1, 0, 0]), color='blue', name="X")
    add_arrow(fig, start=np.array([0, 0, 0]), direction=np.array([0, 1, 0]), color='red', name="Y")
    add_arrow(fig, start=np.array([0, 0, 0]), direction=np.array([0, 0, 1]), color='green', name="Z")

    # Plot spheres with transparency
    for i_sphere, sphere in enumerate(assembly.spheres):
        x, y, z, colors = create_sphere(
            radius=sphere.radius(dofs_array, params_array),
            center=sphere.position(dofs_array, params_array),
            orientation=sphere.orientation(dofs_array, params_array)
        )
        fig.add_trace(go.Surface(
            x=x, y=y, z=z, surfacecolor=colors,
            showscale=False, 
            cmin=0, cmax=assembly.Nspheres / (1 + i_sphere),  
            colorscale='Thermal',  # Change colormap to 'Cividis'
        ))

    # Improve layout: bigger figure, remove axes, center spheres
    fig.update_layout(
        title="3D Sphere Assembly",
        autosize=False,  # Disable autosizing
        width=900,  # Set width in pixels
        height=900,  # Set height in pixels  
        scene=dict(     
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        #     camera=dict(eye=dict(x=1.5, y=0.5, z=0.5))  # Better viewing angle
        )
    )

    fig.show()    

### Plot with sliders

In [ ]:
# Creating sliders
x_slider = widgets.FloatSlider(min=-1, max=1, step=0.1, value=0, description='x')
myradius0_slider = widgets.FloatSlider(min=0, max=1, step=0.1, value=1, description='myradius0')
myradius1_slider = widgets.FloatSlider(min=0, max=1, step=0.1, value=0.5, description='myradius1')

# Create initial figure
fig = go.Figure()

# Wrapper function to extract slider values
def update_wrapper(x, myradius0, myradius1):
    dofs = np.array([x])
    params = np.array([myradius0, myradius1])
    update_figure(fig, myplankton, dofs, params)

# Use interactive widget
interactive_plot = widgets.interactive(update_wrapper, x=x_slider, myradius0=myradius0_slider, myradius1=myradius1_slider)

# Display the interactive widget
display(interactive_plot)

interactive(children=(FloatSlider(value=0.0, description='x', max=1.0, min=-1.0), FloatSlider(value=1.0, descr…

## Fast dynamics

In the limit of fast dynamics of d.o.f., such that we have $\dot{\mathbf{q}}= 0$, the mobility equation reduces formally to two independent mobility equation

$$\mathbf{u}_0 - \mathbf{u}_\infty= \frac{1}{\mu}\mathbb{M}_{k}(\mathbf{q})\cdot\mathbf{F}  + \mathbb{G}_{k}(\mathbf{q})\cdot\mathbf{S}_\infty ,$$
$$\mathbf{q}   =  \mathbb{M}_{dof}(\mathbf{q})\cdot\mathbf{F} + \mathbb{G}_{dof}(\mathbf{q})\cdot\mathbf{S}_\infty.$$

The matrices $\mathbb{M}_{k}(\mathbf{q})$, $\mathbb{G}_{k}(\mathbf{q})$, $\mathbb{M}_{dof}(\mathbf{q})$ and $\mathbb{G}_{dof}(\mathbf{q})$ are non-linear functions in general. 

If the only external force is buyoancy applied at the sphere centers, such that $\tilde{\mathbf{F}} = \mathbf{T}_m\cdot\mathbf{g}$, the problem reduces to

$$\mathbf{u}_0 - \mathbf{u}_\infty= \frac{1}{\mu}\mathbb{M}_{k,m}(\mathbf{q})\cdot\mathbf{g}  + \mathbb{G}_{k}(\mathbf{q})\cdot\mathbf{S}_\infty ,$$
$$ \mathbf{q} =  \mathbb{M}_{dof,m}(\mathbf{q})\cdot\mathbf{g} + \mathbb{G}_{dof}(\mathbf{q})\cdot\mathbf{S}_\infty.$$

(but they can be tabulated provided the dimension of $\mathbf{q}$ is small enough), and

In [9]:
matrices = myplankton.compute_fast_mobility_problem()

print("EQUATION FOR RIGID BODY with gravity")
print("\nUnique Mobility Matrix Mk,m (multiplied by mu):")
print(matrices.Mkm)
print("\nStrain Mobility Matrix Gk (Sxx, Sxy, Sxz, Syy, Syz):")
print(matrices.Gk)

print("\n\nEQUATION FOR DEGREES OF FREEDOM", myplankton.dof_variables)
print("\nMobility Matrix Mdof,m (multiplied by k):")
print(matrices.Mdofm)
print("\nStrain Mobility Matrix Gdof (multiplied by k/mu) (Sxx, Sxy, Sxz, Syy, Syz):")
print(matrices.Gdof)

AttributeError: 'SoftPlankton' object has no attribute 'compute_fast_mobility_problem'

In [ ]:
matrices = myplankton.compute_fast_mobility_problem()

print("EQUATION FOR RIGID BODY with gravity")
print("\nUnique Mobility Matrix Mk,m (multiplied by mu):")
print(matrices.Mkm)
print("\nStrain Mobility Matrix Gk (Sxx, Sxy, Sxz, Syy, Syz):")
print(matrices.Gk)

print("\n\nEQUATION FOR DEGREES OF FREEDOM", myplankton.dof_variables)
print("\nMobility Matrix Mdof,m (multiplied by k):")
print(matrices.Mdofm)
print("\nStrain Mobility Matrix Gdof (multiplied by k/mu) (Sxx, Sxy, Sxz, Syy, Syz):")
print(matrices.Gdof)

EQUATION FOR RIGID BODY with gravity

Unique Mobility Matrix Mk,m (multiplied by mu):
[[ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.021]
 [ 0.    -0.021  0.   ]]

Strain Mobility Matrix Gk (Sxx, Sxy, Sxz, Syy, Syz):
[[ 0.     0.     0.     0.     0.   ]
 [ 0.    -0.     0.     0.     0.   ]
 [ 0.     0.    -0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.    -0.614  0.     0.   ]
 [ 0.     0.614  0.     0.     0.   ]]


EQUATION FOR DEGREES OF FREEDOM ['x']

Mobility Matrix Mdof,m (multiplied by k):
[[-1.  0.  0.]]

Strain Mobility Matrix Gdof (multiplied by k/mu) (Sxx, Sxy, Sxz, Syy, Syz):
[[ 31.416   0.      0.      0.      0.   ]]


## Fluid-structure interaction problem

In [ ]:
class PureShearFlow(Flow):
    """Simple pure shear flow u = (y, 0, 0)."""

    def _velocity(self, pos):
        x, y, z = pos  # Extract coordinates
        shear_rate = self.params[0] if self.params else 1  # Default value
        return jnp.array([shear_rate * y, 0, 0])


In [ ]:
# Create a shear flow with shear rate 1
shear_flow = PureShearFlow(1)

# Test it (steady case)
pos = [1.0, 2.0, 3.0]  
grad_u = shear_flow.gradient(pos)
Omega, S = shear_flow.omega_s(pos)  
print("Velocity Gradient Tensor (∇u):\n", grad_u)
print("Angular velocity Ω:", Omega)
print("Rate-of-strain S):\n", S)


Velocity Gradient Tensor (∇u):
 [[ 0.  1.  0.]
 [ 0.  0.  0.]
 [ 0.  0.  0.]]
Angular velocity Ω: [ 0.   0.  -0.5]
Rate-of-strain S):
 [[ 0.   0.5  0. ]
 [ 0.5  0.   0. ]
 [ 0.   0.   0. ]]


In [ ]:
# parameters
final_time = 500 
dt = 0.5
myplankton.set_param_defaults(new_dict={'k':3, 'myradius0':0.33})

solver = FluidPlankton(myplankton, shear_flow, dt=dt, init_orientation=[1., .2, -.3])
trajectory = solver.simulate(T=final_time)

OLD default param values: ['k', 'myradius0', 'myradius1'] [ 3.    0.33  1.  ]
NEW default param values: ['k', 'myradius0', 'myradius1'] [ 3.    0.33  1.  ]
Time: 0.000 / 500.000  Integrator RK2
Time: 50.000 / 500.000  Integrator RK2
Time: 100.000 / 500.000  Integrator RK2
Time: 150.000 / 500.000  Integrator RK2
Time: 200.000 / 500.000  Integrator RK2
Time: 250.000 / 500.000  Integrator RK2
Time: 300.000 / 500.000  Integrator RK2
Time: 350.000 / 500.000  Integrator RK2
Time: 400.000 / 500.000  Integrator RK2
Time: 450.000 / 500.000  Integrator RK2


In [ ]:
# Extract DOFs (assuming it's the first element in each tuple)
dofs = jnp.array([float(t[0]) for t in trajectory])
positions = jnp.array([t[1] for t in trajectory])
orientations = jnp.array([t[2] for t in trajectory])

# Time vector
t = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=t, y=dofs, mode='lines', name='DOF'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=t, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=t, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()